### Similarity Approach Analysis - No Deviation Episodes

**Approach**: Build lookup table using only episodes WITHOUT deviation (Hasdeviation=False)

This notebook uses the alarm episodes benchmark file to filter and select only "good" episodes (those without deviation) for building the context lookup table. The hypothesis is that learning from cleaner episodes will result in better recommendations.

### Load Data

In [1]:
import pandas as pd
import numpy as np

episodes_operated_tags_df = pd.read_excel('/home/h604827/ControlActions/RESULTS/episode_all_operator_action_plots/episodes_all_with_actions_and_deviations.xlsx')
ssd_df = pd.read_csv('/home/h604827/ControlActions/DATA/SSD_1071_650episodes_1April2026.csv')
events_df = pd.read_csv('/home/h604827/ControlActions/DATA/trip_filtered_events.csv')
pv_op_data_df = pd.read_parquet('/home/h604827/ControlActions/DATA/03LIC_1071_JAN_2026_filtered.parquet')
operating_limits_df = pd.read_csv('/home/h604827/ControlActions/DATA/operating_limits.csv')

# Load benchmark Excel to filter episodes by Hasdeviation
benchmark_df = pd.read_excel('/home/h604827/ControlActions/RESULTS/alarm_episodes_benchmark.xlsx')
print(f"Benchmark data loaded: {len(benchmark_df)} episodes")
print(f"Hasdeviation distribution:\n{benchmark_df['Hasdeviation'].value_counts()}")

# Filter to get episodes without deviation (Hasdeviation=False)
no_deviation_episode_ids = benchmark_df[benchmark_df['Hasdeviation'] == False]['EpisodeID'].tolist()
print(f"\nEpisodes without deviation: {len(no_deviation_episode_ids)}")

episodes_operated_tags_df.head()

Benchmark data loaded: 609 episodes
Hasdeviation distribution:
Hasdeviation
False    404
True     205
Name: count, dtype: int64

Episodes without deviation: 404


,EpisodeID,AlarmStart,AlarmEnd,AlarmDurationMinutes,TotalWindowMinutes,OperatedTags,OperatedTagsCount,DeviatedTags,DeviatedTagsCount,HasOperatorActions,HasOnlyTargetTags,Has1071Action,Has1016Action,Has1013Action
0,1,2022-01-05 08:53:00,2022-01-05 09:33:00,40,186,"03FIC_1085, 03FIC_3435, 03HIC_1141, 03HIC_1151...",15,"02FI_1000.PV, 03FIC_1085.PV, 03FIC_3415.PV, 03...",22,True,False,True,True,True
1,2,2022-01-07 09:55:00,2022-01-07 10:00:00,5,151,"03FIC_3435, 03GHS_0121A, 03GHS_0121AA, 03GHS_0...",8,"03FIC_1085.PV, 03FIC_3415.PV, 03FI_1141A.PV, 0...",23,True,False,False,False,False
2,3,2022-01-07 13:33:00,2022-01-07 13:36:00,3,149,"03FIC_3435, 03LIC_1034",2,"03FIC_1085.PV, 03FIC_3415.PV, 03FI_1141A.PV, 0...",19,True,False,False,False,False
3,4,2022-01-07 14:17:00,2022-01-07 14:19:00,2,148,"03FIC_3435, 03LIC_1016, 03LIC_1034",3,"03FIC_1085.PV, 03FI_1141A.PV, 03LIC_1016.PV, 0...",22,True,False,False,True,False
4,5,2022-01-07 14:54:00,2022-01-07 14:58:00,4,150,"03FIC_3435, 03LIC_1016",2,"02FI_1000.PV, 03FIC_1085.PV, 03FI_1141A.PV, 03...",19,True,False,False,True,False


## Similarity Approach: Building Context for Operator Actions (No Deviation Episodes)

### Approach Overview
1. Use **only episodes without deviation** (Hasdeviation=False from benchmark Excel)
2. For each operator action on the target tags (**03LIC_1071**, **03LIC_1016**, **03PIC_1013**), create a "context" capturing process state
3. Context window: From deviation start time to operator action timestamp

### Rationale
Episodes without deviation represent "good" alarm episodes where the process did not show abnormal behavior patterns. Using these episodes for the lookup table ensures we learn from cleaner, more representative operator actions.

### Context Features (for each of 28 PV tags):
1. **Normalized Position** = (PV_current - Lower_Limit) / (Upper_Limit - Lower_Limit) → 0-1 scale within operating limits
2. **Normalized ROC** = (PV_current - PV_start) / (Upper_Limit - Lower_Limit) → change as fraction of operating range
3. **ROC Direction** = 1 if rising, 0 if falling

### Additional Context Features:
4. **Alarm Proximity** = (PV_1071_current - 28.75) / (Upper_1071 - 28.75) → 0 at alarm, 1 at upper limit
5. **Time Progress** = minutes_since_deviation_start / typical_episode_duration → temporal position in episode

### Filtering No-Deviation Episodes for Lookup Table

Training episodes are selected from episodes where `Hasdeviation=False` (no deviation).
Test episodes are sampled from 2025 (50 random episodes from all episodes in 2025).

In [2]:
# Step 1: Filter to only use episodes WITHOUT deviation for the lookup table
all_episodes_df = episodes_operated_tags_df.copy()
print(f"Total episodes available: {len(all_episodes_df)}")

# Filter to keep only episodes without deviation (Hasdeviation=False)
no_deviation_episodes_df = all_episodes_df[all_episodes_df['EpisodeID'].isin(no_deviation_episode_ids)].copy()
print(f"Episodes without deviation (for lookup table): {len(no_deviation_episodes_df)}")

# Select 50 random episodes from 2025 for testing (from ALL episodes, not just no-deviation)
episodes_2025 = all_episodes_df[
    (all_episodes_df['AlarmStart'] >= pd.Timestamp('2025-01-01')) &
    (all_episodes_df['AlarmStart'] < pd.Timestamp('2026-01-01'))
].copy()

print(f"All episodes in 2025: {len(episodes_2025)}")

sample_n = 50
np.random.seed(42)
if len(episodes_2025) >= sample_n:
    test_episode_ids = episodes_2025.sample(n=sample_n, random_state=42)['EpisodeID'].tolist()
else:
    test_episode_ids = episodes_2025['EpisodeID'].tolist()
    print(f"Warning: Only {len(test_episode_ids)} episodes found in 2025. Using all available.")

# Training episodes: no-deviation episodes excluding test episodes
train_episodes_df = no_deviation_episodes_df[~no_deviation_episodes_df['EpisodeID'].isin(test_episode_ids)].copy()
print(f"\nTest episodes (2025): {len(test_episode_ids)}")
print(f"Training episodes (no deviation, excl. test): {len(train_episodes_df)}")

Total episodes available: 609
Episodes without deviation (for lookup table): 404
All episodes in 2025: 92

Test episodes (2025): 50
Training episodes (no deviation, excl. test): 369


In [ ]:
# AlarmStart values for the selected test episodes
test_alarm_starts_df = (
    episodes_operated_tags_df.loc[
        episodes_operated_tags_df['EpisodeID'].isin(test_episode_ids),
        ['EpisodeID', 'AlarmStart']
    ]
    .drop_duplicates(subset=['EpisodeID'])
    .sort_values('EpisodeID')
    .reset_index(drop=True)
)

print(f"Found AlarmStart for {len(test_alarm_starts_df)} / {len(test_episode_ids)} test episodes")
missing_ids = sorted(set(test_episode_ids) - set(test_alarm_starts_df['EpisodeID']))
if missing_ids:
    print("Missing EpisodeIDs:", missing_ids)

test_alarm_starts_df

Found AlarmStart for 50 / 50 test episodes


,EpisodeID,AlarmStart
0,518,2025-01-03 06:47:00
1,521,2025-01-05 07:16:00
2,522,2025-01-05 08:24:00
3,523,2025-01-06 03:15:00
4,525,2025-01-07 01:12:00
5,527,2025-01-08 03:11:00
6,528,2025-01-08 04:00:00
7,529,2025-01-08 04:26:00
8,530,2025-01-08 05:14:00
9,531,2025-01-08 06:34:00


### Defining the PV columns to be used for context creation

In [6]:
# Step 3: Define the PV columns for context (28 tags)
context_pv_tags = [col for col in pv_op_data_df.columns if col.endswith('.PV')]

# Step 4: Prepare events data for timestamp parsing
events_df['VT_Start'] = pd.to_datetime(events_df['VT_Start'])

# Prepare SSD data for timestamp parsing
ssd_df['AlarmStart_rounded'] = pd.to_datetime(ssd_df['AlarmStart_rounded'])
ssd_df['AlarmEnd_rounded'] = pd.to_datetime(ssd_df['AlarmEnd_rounded'])
ssd_df['First_Transition_Start_Time'] = pd.to_datetime(ssd_df['First_Transition_Start_Time'])

# Prepare episodes data
episodes_operated_tags_df['AlarmStart'] = pd.to_datetime(episodes_operated_tags_df['AlarmStart'])
episodes_operated_tags_df['AlarmEnd'] = pd.to_datetime(episodes_operated_tags_df['AlarmEnd'])

print("Timestamp columns converted to datetime")

Timestamp columns converted to datetime


In [8]:
# Step 5: Helper function to get deviation start time for an episode
# Deviation start is the earliest First_Transition_Start_Time for the target tag (03LIC_1071)
# in that alarm episode

def get_deviation_start_for_episode(episode_id, alarm_start, alarm_end):
    """
    Get the deviation start time for an episode.
    This is the First_Transition_Start_Time for 03LIC_1071 (or earliest among related tags)
    """
    # Find SSD records for this alarm episode (matching by alarm start time)
    episode_ssd = ssd_df[
        (ssd_df['AlarmStart_rounded'] == alarm_start)
    ]
    
    if len(episode_ssd) == 0:
        # Try with a small time tolerance (within 1 minute)
        print(f"No exact SSD data found for EpisodeID {episode_id} with AlarmStart {alarm_start}. Trying with time tolerance.")
        episode_ssd = ssd_df[
            (abs((ssd_df['AlarmStart_rounded'] - alarm_start).dt.total_seconds()) <= 60)
        ]
    
    if len(episode_ssd) == 0:
        # If no SSD data found, use alarm start minus 30 minutes as default deviation start
        print(f"No SSD data found for EpisodeID {episode_id} with AlarmStart {alarm_start}. Using default deviation start.")
        return alarm_start - pd.Timedelta(minutes=30)
    
    # Get deviation start for target tag 03LIC_1071 if available
    target_ssd = episode_ssd[episode_ssd['Tag'] == '03LIC_1071']
    if len(target_ssd) > 0:
        return target_ssd['First_Transition_Start_Time'].iloc[0]
    
    # If not found, use earliest transition start among all tags
    return episode_ssd['First_Transition_Start_Time'].min()

# Test with first available episode
test_ep = all_episodes_df.iloc[0]
test_dev_start = get_deviation_start_for_episode(
    test_ep['EpisodeID'], 
    test_ep['AlarmStart'], 
    test_ep['AlarmEnd']
 )
print(f"Episode {test_ep['EpisodeID']}:")
print(f"  Alarm Start: {test_ep['AlarmStart']}")
print(f"  Deviation Start: {test_dev_start}")
print(f"  Alarm End: {test_ep['AlarmEnd']}")

Episode 1:
  Alarm Start: 2022-01-05 08:53:00
  Deviation Start: 2022-01-05 07:29:00
  Alarm End: 2022-01-05 09:33:00


In [9]:
# Step 6: Helper function to get operator actions for an episode
def get_operator_actions_for_episode(alarm_start, alarm_end, target_sources=['03LIC_1071', '03LIC_1016', '03PIC_1013']):
    """
    Get all CHANGE events (operator actions) for target sources during an episode.
    Episode window: deviation_start to alarm_end
    Only keep SP/OP actions (exclude MODE).
    """
    # Get deviation start
    deviation_start = get_deviation_start_for_episode(None, alarm_start, alarm_end)
    
    # Filter CHANGE events within the episode window for target sources
    actions = events_df[
        (events_df['ConditionName'] == 'CHANGE') &
        (events_df['Source'].isin(target_sources)) &
        (events_df['VT_Start'] >= deviation_start) &
        (events_df['VT_Start'] <= alarm_end)
    ].copy()
    
    # Exclude MODE actions if Description is available
    if 'Description' in actions.columns:
        actions = actions[actions['Description'].isin(['SP', 'OP'])].copy()
    
    return actions, deviation_start

# Test with first selected episode
test_actions, test_dev_start = get_operator_actions_for_episode(
    test_ep['AlarmStart'], 
    test_ep['AlarmEnd']
 )
print(f"Episode {test_ep['EpisodeID']} - Found {len(test_actions)} operator actions")
if len(test_actions) > 0:
    print(test_actions[['Source', 'VT_Start', 'Value', 'PrevValue']].head())

Episode 1 - Found 66 operator actions
            Source                   VT_Start    Value PrevValue
147288  03LIC_1016 2022-01-05 08:42:17.358300  20.0000   37.0000
147289  03LIC_1016 2022-01-05 08:42:17.358300  20.0000       NaN
147296  03PIC_1013 2022-01-05 08:42:23.715200  78.3480       NaN
147297  03PIC_1013 2022-01-05 08:42:23.715200  78.3480   80.3480
147299  03PIC_1013 2022-01-05 08:42:24.781600  76.3480   78.3480


### Sub-Minute Action Merging

**Problem**: Operators often make incremental changes (e.g., turning a knob) that produce multiple CHANGE events within the same minute. Since PV/OP data is minute-wise, all these sub-minute actions receive **identical context vectors**.

**Solution**: Group all actions on the **same tag** within the **same minute** into a single composite action:
- `prev_value` = first action's PrevValue (starting point)
- `value` = last action's Value (ending point)  
- `magnitude` = value - prev_value (net change = operator's full intent)
- `timestamp` = the minute-floor timestamp (matches PV data resolution)

This is not a heuristic — it's a necessary correction for the data resolution mismatch.

In [10]:
# Step 6b: Sub-minute action merging function

def merge_subminute_actions(actions_df):
    """
    Merge actions on the same tag within the same minute into a single composite action.
    
    Since PV/OP data is minute-wise, sub-minute actions all get the same context.
    This function collapses them into the operator's net intent.
    
    Input: DataFrame with columns [Source, VT_Start, Value, PrevValue, Description, ...]
    Output: DataFrame with one row per (Source, minute) — same schema as input but with
            PrevValue = first action's PrevValue, Value = last action's Value,
            VT_Start = minute-floored timestamp
    """
    if len(actions_df) == 0:
        return actions_df.copy()
    
    actions = actions_df.copy()
    
    # Standard dedup: drop rows without PrevValue, parse numerics
    actions['Value_num'] = pd.to_numeric(actions['Value'], errors='coerce')
    actions['PrevValue_num'] = pd.to_numeric(actions['PrevValue'], errors='coerce')
    actions = actions.dropna(subset=['Value_num', 'PrevValue_num'])
    
    if len(actions) == 0:
        return pd.DataFrame(columns=actions_df.columns)
    
    # Remove exact duplicates (same timestamp + source + value)
    actions = actions.drop_duplicates(subset=['VT_Start', 'Source', 'Value'])
    
    # Floor to minute for grouping
    actions['minute_floor'] = actions['VT_Start'].dt.floor('min')
    actions = actions.sort_values(['Source', 'VT_Start'])
    
    merged_records = []
    for (source, minute), group in actions.groupby(['Source', 'minute_floor']):
        group_sorted = group.sort_values('VT_Start')
        
        first_row = group_sorted.iloc[0]
        last_row = group_sorted.iloc[-1]
        
        # Build composite action: first PrevValue -> last Value
        merged = {
            'Source': source,
            'VT_Start': minute,  # Use floored minute (matches PV data)
            'Value': last_row['Value_num'],
            'PrevValue': first_row['PrevValue_num'],
            'Value_num': last_row['Value_num'],
            'PrevValue_num': first_row['PrevValue_num'],
            'ConditionName': 'CHANGE',
            'Description': group_sorted['Description'].mode().iloc[0] if 'Description' in group_sorted.columns else None,
            'num_raw_actions': len(group_sorted),
        }
        merged_records.append(merged)
    
    result = pd.DataFrame(merged_records)
    result['magnitude'] = result['Value_num'] - result['PrevValue_num']
    
    return result

# Quick test: merge the test episode's actions
test_actions_merged = merge_subminute_actions(test_actions)
print(f"Test episode actions: {len(test_actions)} raw -> {len(test_actions_merged)} after sub-minute merging")
if len(test_actions_merged) > 0:
    multi_step = test_actions_merged[test_actions_merged['num_raw_actions'] > 1]
    print(f"  Single-action minutes: {(test_actions_merged['num_raw_actions'] == 1).sum()}")
    print(f"  Multi-action minutes merged: {len(multi_step)}")
    if len(multi_step) > 0:
        print(f"\n  Examples of merged actions:")
        for _, m in multi_step.head(3).iterrows():
            print(f"    {m['Source']} @ {m['VT_Start']}: {m['PrevValue']:.1f} -> {m['Value']:.1f} "
                  f"(net={m['magnitude']:+.1f}, {m['num_raw_actions']} raw actions merged)")


Test episode actions: 66 raw -> 27 after sub-minute merging
  Single-action minutes: 21
  Multi-action minutes merged: 6

  Examples of merged actions:
    03LIC_1016 @ 2022-01-05 09:08:00: 27.0 -> 23.0 (net=-4.0, 2 raw actions merged)
    03LIC_1016 @ 2022-01-05 09:13:00: 23.0 -> 27.0 (net=+4.0, 2 raw actions merged)
    03LIC_1071 @ 2022-01-05 09:04:00: 20.0 -> 24.0 (net=+4.0, 2 raw actions merged)


In [11]:
# Step 7: Helper function to get PV value at a specific timestamp (with nearest lookup)
def get_pv_at_timestamp(timestamp, pv_tag):
    """
    Get PV value at or nearest to the given timestamp.
    Uses forward fill to get the most recent value if exact time not found.
    """
    try:
        # Make timestamp timezone naive if needed
        if timestamp.tzinfo is not None:
            timestamp = timestamp.tz_localize(None)
        
        # Try exact lookup first
        if timestamp in pv_op_data_df.index:
            return pv_op_data_df.loc[timestamp, pv_tag]
        
        # Use asof for nearest lookup (gets value at or before timestamp)
        idx = pv_op_data_df.index.get_indexer([timestamp], method='ffill')[0]
        if idx >= 0 and idx < len(pv_op_data_df):
            return pv_op_data_df.iloc[idx][pv_tag]
        
        # If no value found before, get nearest after
        idx = pv_op_data_df.index.get_indexer([timestamp], method='bfill')[0]
        if idx >= 0 and idx < len(pv_op_data_df):
            return pv_op_data_df.iloc[idx][pv_tag]
        
        return np.nan
    except Exception as e:
        print(f"Error getting PV value for {pv_tag} at {timestamp}: {e}")
        return np.nan

# Test
test_ts = test_dev_start
test_tag = '03LIC_1071.PV'
test_val = get_pv_at_timestamp(test_ts, test_tag)
print(f"PV value for {test_tag} at {test_ts}: {test_val}")

PV value for 03LIC_1071.PV at 2022-01-05 07:29:00: 34.68777


### Function for building context

In [12]:
# Step 8: Build operating limits lookup and context builder

# Build operating limits lookup dict: tag_base -> (lower, upper, range)
op_limits = {}
for _, row in operating_limits_df.iterrows():
    tag_name = row['TAG_NAME']  # e.g. '03LIC_1071.PV'
    tag_base = tag_name.replace('.PV', '').replace('.OP', '')
    lower = row['LOWER_LIMIT']
    upper = row['UPPER_LIMIT']
    op_range = upper - lower
    if op_range > 0:  # Only use tags with valid range
        op_limits[tag_base] = {'lower': lower, 'upper': upper, 'range': op_range}

print(f"Operating limits loaded for {len(op_limits)} tags")

# Alarm threshold for target tag
ALARM_THRESHOLD = 28.75
TARGET_TAG = '03LIC_1071'
target_upper = op_limits[TARGET_TAG]['upper'] if TARGET_TAG in op_limits else 42.41

# Typical episode duration (median) for time progress normalization - will be computed from training data later
TYPICAL_EPISODE_DURATION_MINUTES = 60  # placeholder, updated after loading episodes

def build_context_for_action(deviation_start, action_timestamp, pv_tags):
    """
    Build context features for an operator action using IMPROVED metrics.
    
    Returns a dict with the following for each PV tag:
    - {tag}_norm_pos: Normalized position within operating limits = (PV - Lower) / Range
    - {tag}_norm_roc: Normalized ROC = (PV_action - PV_start) / operating_range
    - {tag}_roc_direction: 1 if positive, 0 if negative
    
    Plus global features:
    - alarm_proximity: (PV_1071 - 28.75) / (Upper_1071 - 28.75)
    - time_progress: minutes since deviation start / typical episode duration
    """
    context = {}
    
    for pv_tag in pv_tags:
        tag_name = pv_tag.replace('.PV', '')
        
        # Get PV values
        pv_at_deviation = get_pv_at_timestamp(deviation_start, pv_tag)
        pv_at_action = get_pv_at_timestamp(action_timestamp, pv_tag)
        
        # Get operating limits for this tag
        limits = op_limits.get(tag_name, None)
        
        if limits and pd.notna(pv_at_action):
            # Normalized position: where is this tag within its operating range?
            norm_pos = (pv_at_action - limits['lower']) / limits['range']
        else:
            norm_pos = np.nan
        
        if limits and pd.notna(pv_at_deviation) and pd.notna(pv_at_action):
            # Normalized ROC: change as fraction of operating range
            norm_roc = (pv_at_action - pv_at_deviation) / limits['range']
        else:
            norm_roc = np.nan
        
        # Direction
        if pd.notna(norm_roc):
            roc_direction = 1 if norm_roc >= 0 else 0
        else:
            roc_direction = np.nan
        
        context[f'{tag_name}_norm_pos'] = norm_pos
        context[f'{tag_name}_norm_roc'] = norm_roc
        context[f'{tag_name}_roc_direction'] = roc_direction
    
    # Global feature: Alarm proximity for target tag
    pv_1071 = get_pv_at_timestamp(action_timestamp, '03LIC_1071.PV')
    if pd.notna(pv_1071):
        context['alarm_proximity'] = (pv_1071 - ALARM_THRESHOLD) / (target_upper - ALARM_THRESHOLD)
    else:
        context['alarm_proximity'] = np.nan
    
    # Global feature: Time progress (fraction of typical episode duration)
    time_delta = (action_timestamp - deviation_start).total_seconds() / 60.0  # in minutes
    context['time_progress'] = time_delta / TYPICAL_EPISODE_DURATION_MINUTES
    
    return context

# Test with first action of first episode
if len(test_actions) > 0:
    first_action = test_actions.iloc[0]
    test_context = build_context_for_action(test_dev_start, first_action['VT_Start'], context_pv_tags)
    print(f"\nContext for action at {first_action['VT_Start']} on {first_action['Source']}:")
    print(f"\n03LIC_1071 (target tag):")
    print(f"  Norm Position: {test_context.get('03LIC_1071_norm_pos', 'N/A'):.4f}")
    print(f"  Norm ROC: {test_context.get('03LIC_1071_norm_roc', 'N/A'):.4f}")
    print(f"  ROC direction: {test_context.get('03LIC_1071_roc_direction', 'N/A')}")
    print(f"\nGlobal features:")
    print(f"  Alarm proximity: {test_context.get('alarm_proximity', 'N/A'):.4f}")
    print(f"  Time progress: {test_context.get('time_progress', 'N/A'):.4f}")
    
    # Show a few other tags for comparison
    for tag in ['03LIC_1016', '03PIC_1013']:
        if f'{tag}_norm_pos' in test_context:
            print(f"\n{tag}:")
            print(f"  Norm Position: {test_context[f'{tag}_norm_pos']:.4f}")
            print(f"  Norm ROC: {test_context[f'{tag}_norm_roc']:.4f}")


Operating limits loaded for 26 tags

Context for action at 2022-01-05 08:42:17.358300 on 03LIC_1016:

03LIC_1071 (target tag):
  Norm Position: 2.8421
  Norm ROC: 2.9203
  ROC direction: 1

Global features:
  Alarm proximity: 1.9660
  Time progress: 1.2215

03LIC_1016:
  Norm Position: 1.0311
  Norm ROC: 0.8946

03PIC_1013:
  Norm Position: 1.9053
  Norm ROC: 2.7438


### Building context with training episodes

In [13]:
# Step 9: Process training episodes and build context for each MERGED operator action
from tqdm import tqdm

# First, compute typical episode duration from training episodes for time_progress normalization
train_durations = (train_episodes_df['AlarmEnd'] - train_episodes_df['AlarmStart']).dt.total_seconds() / 60.0
TYPICAL_EPISODE_DURATION_MINUTES = train_durations.median()
print(f"Typical episode duration (median): {TYPICAL_EPISODE_DURATION_MINUTES:.1f} minutes")

context_records = []
target_sources = ['03LIC_1071', '03LIC_1016', '03PIC_1013']

raw_action_count = 0
merged_action_count = 0

print("Processing training episodes (with sub-minute merging)...")
for idx, (_, episode) in enumerate(tqdm(train_episodes_df.iterrows(), total=len(train_episodes_df))):
    episode_id = episode['EpisodeID']
    alarm_start = episode['AlarmStart']
    alarm_end = episode['AlarmEnd']
    
    # Get deviation start
    deviation_start = get_deviation_start_for_episode(episode_id, alarm_start, alarm_end)
    
    # Get operator actions for this episode (only target tags)
    actions_raw, _ = get_operator_actions_for_episode(alarm_start, alarm_end, target_sources)
    
    if len(actions_raw) == 0:
        continue
    
    # *** Merge sub-minute actions ***
    actions = merge_subminute_actions(actions_raw)
    
    if len(actions) == 0:
        continue
    
    raw_action_count += len(actions_raw)
    merged_action_count += len(actions)
    
    # Process each MERGED action
    for _, action in actions.iterrows():
        action_timestamp = action['VT_Start']
        
        # Build context with normalized features
        context = build_context_for_action(deviation_start, action_timestamp, context_pv_tags)
        
        # Add episode and action metadata
        context['episode_id'] = episode_id
        context['alarm_start'] = alarm_start
        context['alarm_end'] = alarm_end
        context['deviation_start'] = deviation_start
        context['action_timestamp'] = action_timestamp
        context['action_source'] = action['Source']
        context['action_type'] = action['Description']  # SP or OP
        context['action_value'] = action['Value']
        context['action_prev_value'] = action['PrevValue']
        context['action_magnitude'] = action['magnitude']
        context['action_direction'] = 1 if action['magnitude'] > 0 else 0
        context['num_raw_actions'] = action['num_raw_actions']
        
        context_records.append(context)

print(f"\nRaw actions collected: {raw_action_count}")
print(f"After sub-minute merging: {merged_action_count}")
print(f"Reduction: {raw_action_count - merged_action_count} actions merged away ({(1 - merged_action_count/raw_action_count)*100:.1f}%)")
print(f"Total context records created: {len(context_records)}")

# Show action type distribution
temp_df = pd.DataFrame(context_records)
print(f"\nAction type (SP/OP) distribution:")
print(temp_df['action_type'].value_counts())

Typical episode duration (median): 5.0 minutes
Processing training episodes (with sub-minute merging)...


  0%|          | 0/369 [00:00<?, ?it/s]

100%|██████████| 369/369 [00:30<00:00, 12.27it/s]


Raw actions collected: 1882
After sub-minute merging: 361
Reduction: 1521 actions merged away (80.8%)
Total context records created: 361

Action type (SP/OP) distribution:
action_type
OP    314
SP     47
Name: count, dtype: int64


In [14]:
# Step 10: Convert to DataFrame and examine structure
context_df = pd.DataFrame(context_records)

print(f"Context DataFrame shape: {context_df.shape}")
print(f"Total columns: {len(context_df.columns)}")

# Identify new feature columns
norm_pos_cols = [c for c in context_df.columns if '_norm_pos' in c]
norm_roc_cols = [c for c in context_df.columns if '_norm_roc' in c]
dir_cols = [c for c in context_df.columns if '_roc_direction' in c]

print(f"\nColumn breakdown:")
print(f"  - NormPos columns: {len(norm_pos_cols)}")
print(f"  - NormROC columns: {len(norm_roc_cols)}")
print(f"  - Direction columns: {len(dir_cols)}")
print(f"  - Global features: alarm_proximity, time_progress")
print(f"  - Metadata columns: episode_id, alarm_start, alarm_end, deviation_start, action_timestamp, action_source, action_value, action_prev_value, action_magnitude, action_direction")

# Show first few rows with key columns
key_cols = ['episode_id', 'action_timestamp', 'action_source', 'action_direction', 'action_magnitude',
            'alarm_proximity', 'time_progress',
            '03LIC_1071_norm_pos', '03LIC_1071_norm_roc', '03LIC_1071_roc_direction']
context_df[key_cols].head(10)


Context DataFrame shape: (361, 98)
Total columns: 98

Column breakdown:
  - NormPos columns: 28
  - NormROC columns: 28
  - Direction columns: 28
  - Global features: alarm_proximity, time_progress
  - Metadata columns: episode_id, alarm_start, alarm_end, deviation_start, action_timestamp, action_source, action_value, action_prev_value, action_magnitude, action_direction


,episode_id,action_timestamp,action_source,action_direction,action_magnitude,alarm_proximity,time_progress,03LIC_1071_norm_pos,03LIC_1071_norm_roc,03LIC_1071_roc_direction
0,5,2022-01-07 14:55:00,03LIC_1016,1,1.0000,-0.075430,17.4,-1.050699,-1.481350,0
1,29,2022-02-10 15:22:00,03LIC_1071,1,1.0000,-0.031328,17.4,-0.966603,-0.811458,0
2,51,2022-03-04 11:49:00,03PIC_1013,1,2.0000,-0.093662,18.8,-1.085466,-1.641074,0
3,52,2022-03-25 22:31:00,03LIC_1071,1,2.0000,0.204536,10.6,-0.516842,-0.024683,0
4,52,2022-03-25 23:05:00,03LIC_1071,1,2.0000,-0.054079,17.4,-1.009986,-0.517827,0
5,52,2022-03-25 23:06:00,03LIC_1071,1,2.0000,-0.080225,17.6,-1.059842,-0.567684,0
6,52,2022-03-25 22:55:00,03PIC_1013,1,2.0000,0.406848,15.4,-0.131061,0.361098,1
7,56,2022-04-02 06:57:00,03LIC_1071,1,1.1946,-0.027481,17.0,-0.959267,-0.821421,0
8,56,2022-04-02 06:46:00,03PIC_1013,1,2.0000,0.613929,14.8,0.263815,0.401661,1
9,56,2022-04-02 06:53:00,03PIC_1013,1,2.0000,0.295628,16.2,-0.343142,-0.205297,0


In [15]:
# Step 11: Data quality check and summary statistics
print("=== Context DataFrame Summary ===\n")

# Count actions per episode
actions_per_episode = context_df.groupby('episode_id').size()
print(f"Actions per episode:")
print(f"  Min: {actions_per_episode.min()}, Max: {actions_per_episode.max()}, Mean: {actions_per_episode.mean():.1f}")

# Actions by source
print(f"\nActions by target tag:")
print(context_df['action_source'].value_counts())

# Action direction distribution
print(f"\nAction direction distribution:")
print(context_df['action_direction'].value_counts())

# Missing values check
missing_cols = context_df.isnull().sum()
cols_with_missing = missing_cols[missing_cols > 0]
if len(cols_with_missing) > 0:
    print(f"\nColumns with missing values: {len(cols_with_missing)}")
else:
    print(f"\nNo missing values in context features!")

=== Context DataFrame Summary ===

Actions per episode:
  Min: 1, Max: 32, Mean: 4.2

Actions by target tag:
action_source
03PIC_1013    286
03LIC_1071     54
03LIC_1016     21
Name: count, dtype: int64

Action direction distribution:
action_direction
0    196
1    165
Name: count, dtype: int64

Columns with missing values: 12


In [16]:
# Step 12: Clean context records
# Sub-minute merging already handles NaN PrevValue and dedup, but we still filter zero-magnitude actions
print(f"Total records before cleaning: {len(context_df)}")

# Filter out records where action_magnitude is NaN
context_df_clean = context_df[context_df['action_magnitude'].notna()].copy()
print(f"After removing NaN magnitude: {len(context_df_clean)}")

# Remove any remaining exact duplicates (same episode, timestamp, source)
context_df_clean = context_df_clean.drop_duplicates(
    subset=['episode_id', 'action_timestamp', 'action_source']
)
print(f"After removing duplicates: {len(context_df_clean)}")

# Summary
print(f"\nActions per episode:")
actions_per_episode_clean = context_df_clean.groupby('episode_id').size()
print(f"  Min: {actions_per_episode_clean.min()}, Max: {actions_per_episode_clean.max()}, Mean: {actions_per_episode_clean.mean():.1f}")

print(f"\nActions by target tag:")
print(context_df_clean['action_source'].value_counts())

# Show how many were multi-step merges
if 'num_raw_actions' in context_df_clean.columns:
    multi = context_df_clean[context_df_clean['num_raw_actions'] > 1]
    print(f"\nMerged actions (>1 raw action per entry): {len(multi)} ({len(multi)/len(context_df_clean)*100:.1f}%)")
    print(f"Single-step actions: {(context_df_clean['num_raw_actions'] == 1).sum()}")
    print(f"Max raw actions merged into one: {context_df_clean['num_raw_actions'].max()}")

Total records before cleaning: 361
After removing NaN magnitude: 361
After removing duplicates: 361

Actions per episode:
  Min: 1, Max: 32, Mean: 4.2

Actions by target tag:
action_source
03PIC_1013    286
03LIC_1071     54
03LIC_1016     21
Name: count, dtype: int64

Merged actions (>1 raw action per entry): 129 (35.7%)
Single-step actions: 232
Max raw actions merged into one: 36


In [17]:
# Step 13: Save the context DataFrame (with sub-minute merging)
import os, json

output_dir_v3 = '/home/h604827/ControlActions/RESULTS/similarity_test_results/no_deviation_episodes_v3_merged'
os.makedirs(output_dir_v3, exist_ok=True)

output_path = f'{output_dir_v3}/similarity_context_training_merged.csv'
context_df_clean.to_csv(output_path, index=False)
print(f"Context data saved to: {output_path}")
print(f"  Shape: {context_df_clean.shape}")

episodes_info = {
    'context_episodes': train_episodes_df['EpisodeID'].tolist(),
    'total_no_deviation_episodes': len(no_deviation_episode_ids),
    'training_episodes_count': len(train_episodes_df),
    'approach': 'no_deviation_episodes_v3_subminute_merged'
}
with open(f'{output_dir_v3}/context_metadata.json', 'w') as f:
    json.dump(episodes_info, f, indent=2)
print(f"Metadata saved.")

Context data saved to: /home/h604827/ControlActions/RESULTS/similarity_test_results/no_deviation_episodes_v3_merged/similarity_context_training_merged.csv
  Shape: (361, 98)
Metadata saved.


In [18]:
# Step 14: Display sample of the final context dataframe
print("=== Final Context DataFrame (Improved Metrics) ===")
print(f"Shape: {context_df_clean.shape}")

# Show new feature columns
norm_pos_cols = [c for c in context_df_clean.columns if '_norm_pos' in c]
norm_roc_cols = [c for c in context_df_clean.columns if '_norm_roc' in c]
dir_cols = [c for c in context_df_clean.columns if '_roc_direction' in c]

print(f"\nNormPos columns ({len(norm_pos_cols)}): {norm_pos_cols[:5]}...")
print(f"NormROC columns ({len(norm_roc_cols)}): {norm_roc_cols[:5]}...")
print(f"Direction columns ({len(dir_cols)}): {dir_cols[:5]}...")

# Show statistics for key features
print(f"\n=== 03LIC_1071 NormPos Statistics ===")
print(context_df_clean['03LIC_1071_norm_pos'].describe())
print(f"\n=== 03LIC_1071 NormROC Statistics ===")
print(context_df_clean['03LIC_1071_norm_roc'].describe())
print(f"\n=== Alarm Proximity Statistics ===")
print(context_df_clean['alarm_proximity'].describe())
print(f"\n=== Time Progress Statistics ===")
print(context_df_clean['time_progress'].describe())


=== Final Context DataFrame (Improved Metrics) ===
Shape: (361, 98)

NormPos columns (28): ['03LIC_1071_norm_pos', '02FI_1000_norm_pos', '03FIC_1085_norm_pos', '03FIC_3415_norm_pos', '03FIC_3435_norm_pos']...
NormROC columns (28): ['03LIC_1071_norm_roc', '02FI_1000_norm_roc', '03FIC_1085_norm_roc', '03FIC_3415_norm_roc', '03FIC_3435_norm_roc']...
Direction columns (28): ['03LIC_1071_roc_direction', '02FI_1000_roc_direction', '03FIC_1085_roc_direction', '03FIC_3415_roc_direction', '03FIC_3435_roc_direction']...

=== 03LIC_1071 NormPos Statistics ===
count    361.000000
mean       0.449162
std        1.598198
min       -4.877026
25%       -0.340032
50%        0.365940
75%        1.120471
max        8.692236
Name: 03LIC_1071_norm_pos, dtype: float64

=== 03LIC_1071 NormROC Statistics ===
count    361.000000
mean      -0.117055
std        1.647767
min       -9.848714
25%       -0.879183
50%       -0.116518
75%        0.548194
max        5.614427
Name: 03LIC_1071_norm_roc, dtype: float64

=

---
## Improved 4-Component Similarity Approach

### Key Changes from Previous Approach
1. **Dropped PV cosine similarity** — it was ~0.9978 everywhere and provided zero signal
2. **Normalized all features by operating range** — makes cross-tag comparisons meaningful
3. **Added alarm proximity** — captures urgency (distance from alarm threshold)
4. **Added temporal progress** — early vs late actions in an episode are qualitatively different
5. **Weighted direction match by movement magnitude** — noisy 0/1 directions on near-static tags no longer dominate

### 4 Similarity Components:
| Component | Distance Metric | Weight | What it captures |
|-----------|----------------|--------|------------------|
| **NormPos** (Normalized Position) | Euclidean | 0.25 | Where the plant is right now relative to operating limits |
| **NormROC** (Normalized Rate of Change) | Euclidean | 0.30 | How fast each tag is changing relative to its range |
| **Alarm Proximity** | Absolute difference | 0.20 | How close 03LIC_1071 is to the alarm threshold |
| **Weighted Direction Match** | Weighted agreement | 0.25 | Are tags moving in the same directions (weighted by magnitude) |

### Testing on Reserved Episodes (Using No-Deviation Lookup Table)

For each test episode:
1. Start from deviation start time
2. For each target-tag SP/OP action timestamp:
   - Calculate context at that timestamp
   - Compare with all training contexts using 4-component similarity
   - Return the most similar historical action(s)

In [19]:
# Step 15: Define column sets for the improved 4-component similarity

# Feature column sets
norm_pos_cols = [c for c in context_df_clean.columns if '_norm_pos' in c]
norm_roc_cols = [c for c in context_df_clean.columns if '_norm_roc' in c]
dir_cols = [c for c in context_df_clean.columns if '_roc_direction' in c]

# Only keep tags that have operating limits (valid normalized features)
valid_tags = set(op_limits.keys())
norm_pos_cols = [c for c in norm_pos_cols if c.replace('_norm_pos', '') in valid_tags]
norm_roc_cols = [c for c in norm_roc_cols if c.replace('_norm_roc', '') in valid_tags]
dir_cols = [c for c in dir_cols if c.replace('_roc_direction', '') in valid_tags]

print(f"Tags with valid operating limits: {len(valid_tags)}")
print(f"NormPos columns: {len(norm_pos_cols)}")
print(f"NormROC columns: {len(norm_roc_cols)}")
print(f"Direction columns: {len(dir_cols)}")

# Pre-check: how much variance do the new features have?
print(f"\n=== Feature Variance Check (should show spread, not constant) ===")
for col_set, label in [(norm_pos_cols, 'NormPos'), (norm_roc_cols, 'NormROC')]:
    stds = context_df_clean[col_set].std()
    print(f"\n{label} std dev per tag (top 5 most variable):")
    for tag, std in stds.nlargest(5).items():
        print(f"  {tag}: std={std:.4f}")
    print(f"  Mean std across all tags: {stds.mean():.4f}")

print(f"\nAlarm proximity std: {context_df_clean['alarm_proximity'].std():.4f}")
print(f"Time progress std: {context_df_clean['time_progress'].std():.4f}")


Tags with valid operating limits: 26
NormPos columns: 26
NormROC columns: 26
Direction columns: 26

=== Feature Variance Check (should show spread, not constant) ===

NormPos std dev per tag (top 5 most variable):
  03FI_1141A_norm_pos: std=424494.7453
  03TIC_1142_norm_pos: std=4.3984
  02FI_1000_norm_pos: std=3.5236
  03TI_1081_norm_pos: std=2.7830
  03TI_1421_norm_pos: std=2.6203
  Mean std across all tags: 16328.3892

NormROC std dev per tag (top 5 most variable):
  03FI_1141A_norm_roc: std=267813.2707
  03TIC_1142_norm_roc: std=3.9347
  02FI_1000_norm_roc: std=2.3843
  03LIC_1071_norm_roc: std=1.6478
  03PIC_3131_norm_roc: std=1.4841
  Mean std across all tags: 10301.5394

Alarm proximity std: 0.8381
Time progress std: 5.3883


In [20]:

# Step 16: Improved 4-Component Similarity Function

def calculate_weighted_similarity(runtime_context, historical_df, 
                                   norm_pos_cols, norm_roc_cols, dir_cols,
                                   w_norm_pos=0.25, w_norm_roc=0.30, 
                                   w_alarm_prox=0.20, w_dir=0.25):
    """
    Calculate 4-component similarity between runtime context and all historical contexts.
    
    Components:
    1. NormPos Similarity (Euclidean distance, converted to similarity)
       - Captures WHERE each tag is relative to its operating limits
    2. NormROC Similarity (Euclidean distance, converted to similarity)
       - Captures HOW FAST each tag is changing relative to its range
    3. Alarm Proximity Similarity (absolute difference, converted to similarity)
       - Captures HOW URGENT the situation is for target tag
    4. Weighted Direction Match (weighted by NormROC magnitude)
       - Captures WHETHER tags are moving in the same direction
       - Weighted so that tags with larger movement count more
    
    Returns: DataFrame with similarity scores and corresponding action details
    """
    similarities = []
    n_tags = len(norm_pos_cols)
    
    # --- Pre-extract runtime vectors ---
    runtime_norm_pos = np.array([runtime_context.get(col, 0) for col in norm_pos_cols])
    runtime_norm_pos = np.nan_to_num(runtime_norm_pos, nan=0.0)
    
    runtime_norm_roc = np.array([runtime_context.get(col, 0) for col in norm_roc_cols])
    runtime_norm_roc = np.nan_to_num(runtime_norm_roc, nan=0.0)
    
    runtime_dir = np.array([runtime_context.get(col, 0) for col in dir_cols])
    runtime_dir = np.nan_to_num(runtime_dir, nan=0.0)
    
    runtime_alarm_prox = runtime_context.get('alarm_proximity', 0.5)
    if pd.isna(runtime_alarm_prox):
        runtime_alarm_prox = 0.5
    
    # --- Pre-extract historical arrays for vectorized computation ---
    hist_norm_pos = historical_df[norm_pos_cols].fillna(0).values  # shape: (N, n_tags)
    hist_norm_roc = historical_df[norm_roc_cols].fillna(0).values
    hist_dir = historical_df[dir_cols].fillna(0).values
    hist_alarm_prox = historical_df['alarm_proximity'].fillna(0.5).values
    
    # --- Component 1: NormPos similarity (Euclidean distance → similarity) ---
    # Max possible Euclidean distance for normalization: sqrt(n_tags) if all tags differ by 1.0
    norm_pos_diffs = hist_norm_pos - runtime_norm_pos  # (N, n_tags)
    norm_pos_dists = np.sqrt(np.sum(norm_pos_diffs ** 2, axis=1))  # (N,)
    max_norm_pos_dist = np.sqrt(n_tags)  # theoretical max
    norm_pos_sim = 1.0 - (norm_pos_dists / max_norm_pos_dist)  # (N,) in [0, 1]
    
    # --- Component 2: NormROC similarity (Euclidean distance → similarity) ---
    norm_roc_diffs = hist_norm_roc - runtime_norm_roc
    norm_roc_dists = np.sqrt(np.sum(norm_roc_diffs ** 2, axis=1))
    # Use a practical max: ROC rarely exceeds ±1.0 per tag, so max dist ≈ sqrt(n_tags * 4)
    max_norm_roc_dist = np.sqrt(n_tags * 4)
    norm_roc_sim = 1.0 - np.clip(norm_roc_dists / max_norm_roc_dist, 0, 1)
    
    # --- Component 3: Alarm proximity similarity (absolute difference → similarity) ---
    alarm_prox_diffs = np.abs(hist_alarm_prox - runtime_alarm_prox)
    # Max possible difference is ~1.0 (at alarm vs at upper limit)
    alarm_prox_sim = 1.0 - np.clip(alarm_prox_diffs, 0, 1)
    
    # --- Component 4: Weighted direction match (weighted by NormROC magnitude) ---
    # Weight each tag's direction match by max(|runtime_roc|, |hist_roc|) so that
    # tags with larger movement contribute more to the match score
    abs_runtime_roc = np.abs(runtime_norm_roc)  # (n_tags,)
    abs_hist_roc = np.abs(hist_norm_roc)  # (N, n_tags)
    movement_weights = np.maximum(abs_runtime_roc, abs_hist_roc)  # (N, n_tags)
    
    dir_matches_matrix = (hist_dir == runtime_dir).astype(float)  # (N, n_tags)
    
    weight_sums = movement_weights.sum(axis=1)  # (N,)
    # Use safe_weight_sums to avoid division-by-zero warnings
    # (np.where evaluates both branches eagerly, so the divide still fires on zero rows)
    safe_weight_sums = np.where(weight_sums > 0, weight_sums, 1.0)
    weighted_dir_match = np.where(
        weight_sums > 0,
        (dir_matches_matrix * movement_weights).sum(axis=1) / safe_weight_sums,
        0.5  # default when no tag is moving
    )
    
    # --- Combined similarity ---
    total_sim = (w_norm_pos * norm_pos_sim + 
                 w_norm_roc * norm_roc_sim + 
                 w_alarm_prox * alarm_prox_sim + 
                 w_dir * weighted_dir_match)
    
    # --- Build results DataFrame ---
    results = pd.DataFrame({
        'hist_index': historical_df.index,
        'total_similarity': total_sim,
        'norm_pos_similarity': norm_pos_sim,
        'norm_roc_similarity': norm_roc_sim,
        'alarm_prox_similarity': alarm_prox_sim,
        'weighted_dir_match': weighted_dir_match,
        'action_source': historical_df['action_source'].values,
        'action_type': historical_df['action_type'].values,
        'action_direction': historical_df['action_direction'].values,
        'action_magnitude': historical_df['action_magnitude'].values,
        'episode_id': historical_df['episode_id'].values,
        'num_raw_actions': historical_df['num_raw_actions'].values
    })
    
    return results.sort_values('total_similarity', ascending=False)

print("Improved 4-component similarity function defined")
print(f"  Component 1: NormPos similarity (Euclidean, weight=0.25)")
print(f"  Component 2: NormROC similarity (Euclidean, weight=0.30)")
print(f"  Component 3: Alarm proximity similarity (abs diff, weight=0.20)")
print(f"  Component 4: Weighted direction match (weight=0.25)")
print(f"\nNote: Fully vectorized — no per-row Python loop")
print(f"Output now includes action_type (SP/OP) and num_raw_actions for each match")


Improved 4-component similarity function defined
  Component 1: NormPos similarity (Euclidean, weight=0.25)
  Component 2: NormROC similarity (Euclidean, weight=0.30)
  Component 3: Alarm proximity similarity (abs diff, weight=0.20)
  Component 4: Weighted direction match (weight=0.25)

Note: Fully vectorized — no per-row Python loop
Output now includes action_type (SP/OP) and num_raw_actions for each match


In [21]:
# Step 17: Function to build context at a specific timestamp for runtime evaluation

def build_runtime_context(deviation_start, current_time, pv_tags):
    """
    Build context at runtime — same structure as training context (improved metrics).
    Uses normalized position, normalized ROC, alarm proximity, and time progress.
    """
    context = {}
    
    for pv_tag in pv_tags:
        tag_base = pv_tag.replace('.PV', '')
        
        pv_at_start = get_pv_at_timestamp(deviation_start, pv_tag)
        pv_at_current = get_pv_at_timestamp(current_time, pv_tag)
        
        limits = op_limits.get(tag_base, None)
        
        # Normalized position
        if limits and pd.notna(pv_at_current):
            norm_pos = (pv_at_current - limits['lower']) / limits['range']
        else:
            norm_pos = 0.0
        
        # Normalized ROC
        if limits and pd.notna(pv_at_start) and pd.notna(pv_at_current):
            norm_roc = (pv_at_current - pv_at_start) / limits['range']
        else:
            norm_roc = 0.0
        
        # ROC direction
        roc_direction = 1 if norm_roc >= 0 else 0
        
        context[f'{tag_base}_norm_pos'] = norm_pos
        context[f'{tag_base}_norm_roc'] = norm_roc
        context[f'{tag_base}_roc_direction'] = roc_direction
    
    # Alarm proximity
    pv_1071 = get_pv_at_timestamp(current_time, '03LIC_1071.PV')
    if pd.notna(pv_1071):
        context['alarm_proximity'] = (pv_1071 - ALARM_THRESHOLD) / (target_upper - ALARM_THRESHOLD)
    else:
        context['alarm_proximity'] = 0.5
    
    # Time progress
    time_delta = (current_time - deviation_start).total_seconds() / 60.0
    context['time_progress'] = time_delta / TYPICAL_EPISODE_DURATION_MINUTES
    
    return context

print("Runtime context builder function defined (improved metrics)")


Runtime context builder function defined (improved metrics)


In [ ]:
# Step 25: Run similarity approach with SUB-MINUTE MERGED actions (2025 test episodes)
# Using NO DEVIATION episodes lookup table + 4-component similarity + sub-minute merging
import os
from tqdm import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Create output directory for v3 (merged) approach
output_dir = '/home/h604827/ControlActions/RESULTS/similarity_test_results/no_deviation_episodes_v3_merged'
os.makedirs(output_dir, exist_ok=True)

print(f"=== 4-Component Similarity + Sub-Minute Merging ===")
print(f"Training lookup table: {len(context_df_clean)} entries from {context_df_clean['episode_id'].nunique()} episodes")
print(f"Test episodes from 2025: {len(test_episode_ids)}")

magnitude_tolerance = 3  # abs(diff) <= this value counts as match

# Store summary and excel rows
all_episodes_summary = []
excel_rows_by_episode = {}

overall_counts = {
    'total_actual': 0,
    'tag_matches': 0,
    'type_matches': 0,
    'dir_matches': 0,
    'mag_matches': 0,
    'all_matches': 0
}

# Define target PV tags (visible by default) and their colors
target_pv_tags_vis = ['03LIC_1071.PV', '03LIC_1016.PV', '03PIC_1013.PV']
target_pv_colors = {'03LIC_1071.PV': 'blue', '03LIC_1016.PV': 'green', '03PIC_1013.PV': 'orange'}

print(f"Processing {len(test_episode_ids)} test episodes...")
print(f"Output directory: {output_dir}\n")

for test_ep_id in tqdm(test_episode_ids, desc="Test episodes"):
    # Get episode details
    ep_data = episodes_operated_tags_df[episodes_operated_tags_df['EpisodeID'] == test_ep_id].iloc[0]
    ep_alarm_start = ep_data['AlarmStart']
    ep_alarm_end = ep_data['AlarmEnd']
    ep_operated_tags = ep_data['OperatedTags']
    
    # Get deviation start
    ep_deviation_start = get_deviation_start_for_episode(test_ep_id, ep_alarm_start, ep_alarm_end)
    
    # *** Get actual actions and MERGE sub-minute ***
    ep_actual_actions_raw, _ = get_operator_actions_for_episode(
        ep_alarm_start, ep_alarm_end,
        target_sources=['03LIC_1071', '03LIC_1016', '03PIC_1013']
    )
    
    if len(ep_actual_actions_raw) == 0:
        continue
    
    ep_actual_clean = merge_subminute_actions(ep_actual_actions_raw)
    
    if len(ep_actual_clean) == 0:
        continue
    
    # Add direction column
    ep_actual_clean['action_direction'] = (ep_actual_clean['magnitude'] > 0).astype(int)
    ep_actual_clean['action_magnitude'] = ep_actual_clean['magnitude']
    # Description column already contains SP/OP from merge function
    ep_actual_clean['action_type'] = ep_actual_clean['Description']
    
    # Run similarity matching at each merged action timestamp
    ep_action_results = []
    for _, act in ep_actual_clean.iterrows():
        current_time = pd.to_datetime(act['VT_Start'])
        
        runtime_ctx = build_runtime_context(ep_deviation_start, current_time, context_pv_tags)
        sim_results = calculate_weighted_similarity(
            runtime_ctx, context_df_clean, norm_pos_cols, norm_roc_cols, dir_cols
        )
        
        for rank, (_, match) in enumerate(sim_results.head(3).iterrows(), 1):
            ep_action_results.append({
                'episode_id': test_ep_id,
                'action_time': current_time,
                'rank': rank,
                'similarity': match['total_similarity'],
                'norm_pos_similarity': match['norm_pos_similarity'],
                'norm_roc_similarity': match['norm_roc_similarity'],
                'alarm_prox_similarity': match['alarm_prox_similarity'],
                'weighted_dir_match': match['weighted_dir_match'],
                'recommended_action_source': match['action_source'],
                'recommended_action_type': match['action_type'],
                'recommended_action_direction': match['action_direction'],
                'recommended_action_magnitude': match['action_magnitude'],
                'recommended_num_raw_actions': int(match['num_raw_actions']),
                'matched_episode_id': match['episode_id'],
                'actual_action_source': act['Source'],
                'actual_action_type': act['action_type'],
                'actual_action_direction': int(act['action_direction']),
                'actual_action_magnitude': float(act['action_magnitude']),
                'actual_num_raw_actions': int(act['num_raw_actions']),
                'alarm_proximity': runtime_ctx['alarm_proximity'],
                'time_progress': runtime_ctx['time_progress'],
                '1071_norm_roc': runtime_ctx.get('03LIC_1071_norm_roc', 0)
            })
    
    if len(ep_action_results) == 0:
        continue
    
    ep_results_df = pd.DataFrame(ep_action_results)
    ep_top_recs = ep_results_df[ep_results_df['rank'] == 1].copy()
    ep_top_recs = ep_top_recs.sort_values('action_time')  # Sort by time to avoid zigzag lines
    
    # Create and save visualization
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        subplot_titles=[
            f'Episode {test_ep_id}: All PV Tags (Normalized to Operating Limits)',
            'Recommended Action Source at Merged Action Times',
            'Similarity Components at Action Times'
        ],
        vertical_spacing=0.08,
        row_heights=[0.40, 0.25, 0.35]
    )
    
    pv_start = ep_deviation_start - pd.Timedelta(minutes=10)
    pv_end = ep_alarm_end + pd.Timedelta(minutes=70)
    
    actual_target_tags_operated = sorted(ep_actual_clean['Source'].unique().tolist())
    actual_target_tags_str = ', '.join(actual_target_tags_operated)
    
    # Row 1: All PV Tags normalized to operating limits (0=Lower, 1=Upper)
    for pv_tag in context_pv_tags:
        tag_base = pv_tag.replace('.PV', '')
        limits = op_limits.get(tag_base, None)
        if limits is None:
            continue
        
        pv_window = pv_op_data_df.loc[pv_start:pv_end, pv_tag]
        if pv_window.empty:
            continue
        
        # Normalize: 0 = lower limit, 1 = upper limit
        norm_vals = (pv_window.values - limits['lower']) / limits['range']
        
        is_target = pv_tag in target_pv_tags_vis
        fig.add_trace(
            go.Scatter(
                x=pv_window.index, y=norm_vals, mode='lines',
                name=pv_tag,
                line=dict(
                    color=target_pv_colors.get(pv_tag, None),
                    width=3 if pv_tag == '03LIC_1071.PV' else (2 if is_target else 1)
                ),
                visible=True if is_target else 'legendonly',
                hovertemplate=f'{pv_tag}<br>Norm: %{{y:.3f}}<br>Raw: %{{customdata:.2f}}<br>Time: %{{x}}',
                customdata=pv_window.values
            ),
            row=1, col=1
        )
    
    # Alarm threshold normalized for 03LIC_1071
    if TARGET_TAG in op_limits:
        alarm_norm = (ALARM_THRESHOLD - op_limits[TARGET_TAG]['lower']) / op_limits[TARGET_TAG]['range']
        fig.add_hline(y=alarm_norm, line_dash="dash", line_color="red",
                      annotation_text=f"1071 Alarm ({ALARM_THRESHOLD})", row=1, col=1)
    
    fig.add_vrect(x0=ep_deviation_start, x1=ep_alarm_start, fillcolor="orange", opacity=0.1, line_width=0, row=1, col=1)
    fig.add_vrect(x0=ep_alarm_start, x1=ep_alarm_end, fillcolor="red", opacity=0.2, line_width=0, row=1, col=1)
    
    # Action markers at 03LIC_1071's normalized position
    action_times = pd.to_datetime(ep_actual_clean['VT_Start'])
    if TARGET_TAG in op_limits:
        action_pv_raw = [get_pv_at_timestamp(t, '03LIC_1071.PV') for t in action_times]
        action_pv_norm = [
            (v - op_limits[TARGET_TAG]['lower']) / op_limits[TARGET_TAG]['range'] if pd.notna(v) else np.nan
            for v in action_pv_raw
        ]
    else:
        action_pv_norm = [get_pv_at_timestamp(t, '03LIC_1071.PV') for t in action_times]
    
    fig.add_trace(
        go.Scatter(x=action_times, y=action_pv_norm, mode='markers', 
                   name=f'Actual Actions',
                   marker=dict(symbol='triangle-up', size=20, color='red'),
                   text=[f"{d} Net: {m:+.1f} ({n} raw)" for d, m, n in zip(ep_actual_clean['action_type'], ep_actual_clean['magnitude'], ep_actual_clean['num_raw_actions'])],
                   hovertemplate='%{text}<br>Time: %{x}'),
        row=1, col=1
    )
    
    # Row 2: Recommended action source at action times
    source_map = {'03LIC_1071': 0, '03LIC_1016': 1, '03PIC_1013': 2}
    ep_top_recs['source_numeric'] = ep_top_recs['recommended_action_source'].map(source_map)
    colors = ['green' if d == 1 else 'red' for d in ep_top_recs['recommended_action_direction']]
    fig.add_trace(
        go.Scatter(x=ep_top_recs['action_time'], y=ep_top_recs['source_numeric'],
                  mode='markers', marker=dict(size=8, color=colors),
                  name='Recommended (green=up, red=down)',
                  text=[f"Tag: {s}<br>Type: {t}<br>Dir: {'up' if d==1 else 'down'}<br>Mag: {mag:+.1f} ({nraw} steps)<br>Sim: {sim:.3f}" 
                        for s, t, d, mag, nraw, sim in zip(ep_top_recs['recommended_action_source'],
                                                     ep_top_recs['recommended_action_type'],
                                                     ep_top_recs['recommended_action_direction'],
                                                     ep_top_recs['recommended_action_magnitude'],
                                                     ep_top_recs['recommended_num_raw_actions'],
                                                     ep_top_recs['similarity'])],
                  hoverinfo='text'),
        row=2, col=1
    )
    fig.add_vrect(x0=ep_alarm_start, x1=ep_alarm_end, fillcolor="red", opacity=0.2, line_width=0, row=2, col=1)
    
    # Row 3: Similarity components
    fig.add_trace(go.Scatter(x=ep_top_recs['action_time'], y=ep_top_recs['similarity'],
                  mode='markers+lines', name='Total Similarity', line=dict(color='purple', width=2)), row=3, col=1)
    fig.add_trace(go.Scatter(x=ep_top_recs['action_time'], y=ep_top_recs['norm_pos_similarity'],
                  mode='markers+lines', name='NormPos Sim', line=dict(color='blue', dash='dot')), row=3, col=1)
    fig.add_trace(go.Scatter(x=ep_top_recs['action_time'], y=ep_top_recs['norm_roc_similarity'],
                  mode='markers+lines', name='NormROC Sim', line=dict(color='green', dash='dot')), row=3, col=1)
    fig.add_trace(go.Scatter(x=ep_top_recs['action_time'], y=ep_top_recs['alarm_prox_similarity'],
                  mode='markers+lines', name='Alarm Prox Sim', line=dict(color='red', dash='dot')), row=3, col=1)
    fig.add_trace(go.Scatter(x=ep_top_recs['action_time'], y=ep_top_recs['weighted_dir_match'],
                  mode='markers+lines', name='Weighted Dir Match', line=dict(color='orange', dash='dot')), row=3, col=1)
    fig.add_vrect(x0=ep_alarm_start, x1=ep_alarm_end, fillcolor="red", opacity=0.2, line_width=0, row=3, col=1)
    
    fig.update_layout(
        height=1100,
        title_text=f'Episode {test_ep_id} - Similarity + Sub-Minute Merging<br>'
                   f'<sub>Actual Tags: {actual_target_tags_str} | Actions: {len(ep_actual_clean)} merged from {ep_actual_clean["num_raw_actions"].sum()} raw</sub>',
        showlegend=True
    )
    fig.update_xaxes(range=[pv_start, pv_end], row=1, col=1)
    fig.update_xaxes(range=[pv_start, pv_end], row=2, col=1)
    fig.update_xaxes(range=[pv_start, pv_end], row=3, col=1)
    fig.update_yaxes(title_text="Normalized Position (0=Lower, 1=Upper Limit)", row=1, col=1)
    fig.update_yaxes(title_text="Tag", ticktext=['1071', '1016', '1013'], tickvals=[0, 1, 2], row=2, col=1)
    fig.update_yaxes(title_text="Similarity Score", row=3, col=1)
    
    html_path = f'{output_dir}/episode_{test_ep_id}_visualization.html'
    fig.write_html(html_path)
    
    # Compute top-1 match metrics and build Excel rows
    rows = []
    tag_matches = 0
    type_matches = 0
    dir_matches = 0
    mag_matches = 0
    all_matches = 0
    total_actual = len(ep_actual_clean)
    
    for i, (_, act) in enumerate(ep_actual_clean.iterrows(), start=1):
        act_time = pd.to_datetime(act['VT_Start'])
        act_source = act['Source']
        act_type = act['action_type']
        act_dir = int(act['action_direction'])
        act_mag = float(act['action_magnitude'])
        
        rows.append({
            'episode_id': test_ep_id,
            'action_group': i,
            'row_type': 'actual',
            'action_time': act_time,
            'source': act_source,
            'action_type': act_type,
            'direction': act_dir,
            'magnitude': act_mag,
            'prev_value': act['PrevValue'],
            'value': act['Value'],
            'num_raw_actions': int(act['num_raw_actions']),
            'similarity': np.nan,
            'matched_episode_id': np.nan,
            'tag_match': np.nan,
            'type_match': np.nan,
            'direction_match': np.nan,
            'magnitude_match': np.nan
        })
        
        recs = ep_results_df[ep_results_df['action_time'] == act_time].sort_values('rank')
        for _, rec in recs.iterrows():
            rec_source = rec['recommended_action_source']
            rec_type = rec['recommended_action_type']
            rec_dir = int(rec['recommended_action_direction'])
            rec_mag = rec['recommended_action_magnitude']
            rec_num_raw = int(rec['recommended_num_raw_actions'])
            
            tag_ok = rec_source == act_source
            type_ok = rec_type == act_type
            dir_ok = rec_dir == act_dir
            mag_ok = pd.notna(rec_mag) and abs(rec_mag - act_mag) <= magnitude_tolerance
            
            rows.append({
                'episode_id': test_ep_id,
                'action_group': i,
                'row_type': f'recommended_rank{int(rec["rank"])}',
                'action_time': act_time,
                'source': rec_source,
                'action_type': rec_type,
                'direction': rec_dir,
                'magnitude': rec_mag,
                'prev_value': np.nan,
                'value': np.nan,
                'num_raw_actions': rec_num_raw,
                'similarity': rec['similarity'],
                'matched_episode_id': rec['matched_episode_id'],
                'tag_match': 1 if tag_ok else 0,
                'type_match': 1 if type_ok else 0,
                'direction_match': 1 if dir_ok else 0,
                'magnitude_match': 1 if mag_ok else 0
            })
        
        # Top-1 match
        top1 = ep_top_recs[ep_top_recs['action_time'] == act_time]
        if len(top1) > 0:
            top1 = top1.iloc[0]
            tag_ok = top1['recommended_action_source'] == act_source
            type_ok = top1['recommended_action_type'] == act_type
            dir_ok = int(top1['recommended_action_direction']) == act_dir
            mag_ok = pd.notna(top1['recommended_action_magnitude']) and abs(top1['recommended_action_magnitude'] - act_mag) <= magnitude_tolerance
            if tag_ok: tag_matches += 1
            if type_ok: type_matches += 1
            if dir_ok: dir_matches += 1
            if mag_ok: mag_matches += 1
            if tag_ok and type_ok and dir_ok and mag_ok: all_matches += 1
    
    excel_rows_by_episode[test_ep_id] = pd.DataFrame(rows)
    
    summary = {
        'episode_id': test_ep_id,
        'alarm_start': str(ep_alarm_start),
        'alarm_end': str(ep_alarm_end),
        'actual_actions_count': total_actual,
        'tag_match_accuracy': tag_matches / total_actual if total_actual > 0 else None,
        'type_match_accuracy': type_matches / total_actual if total_actual > 0 else None,
        'direction_match_accuracy': dir_matches / total_actual if total_actual > 0 else None,
        'magnitude_match_accuracy': mag_matches / total_actual if total_actual > 0 else None,
        'all_match_accuracy': all_matches / total_actual if total_actual > 0 else None
    }
    all_episodes_summary.append(summary)
    overall_counts['total_actual'] += total_actual
    overall_counts['tag_matches'] += tag_matches
    overall_counts['type_matches'] += type_matches
    overall_counts['dir_matches'] += dir_matches
    overall_counts['mag_matches'] += mag_matches
    overall_counts['all_matches'] += all_matches

# Save Excel report
excel_path = f"{output_dir}/operator_action_recommendations_by_episode.xlsx"
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    for ep_id in test_episode_ids:
        if ep_id not in excel_rows_by_episode:
            continue
        sheet_name = f"episode_{ep_id}"
        excel_rows_by_episode[ep_id].to_excel(writer, sheet_name=sheet_name, index=False)

# Save summary
summary_df = pd.DataFrame(all_episodes_summary)
summary_df.to_excel(f"{output_dir}/episode_summary.xlsx", index=False)

overall_total = overall_counts['total_actual']
if overall_total > 0:
    print(f"\n{'='*60}")
    print(f"OVERALL RESULTS (Sub-Minute Merged)")
    print(f"{'='*60}")
    print(f"Tag match:       {overall_counts['tag_matches']}/{overall_total} = {overall_counts['tag_matches']/overall_total*100:.1f}%")
    print(f"Type match:      {overall_counts['type_matches']}/{overall_total} = {overall_counts['type_matches']/overall_total*100:.1f}%")
    print(f"Direction match: {overall_counts['dir_matches']}/{overall_total} = {overall_counts['dir_matches']/overall_total*100:.1f}%")
    print(f"Magnitude match: {overall_counts['mag_matches']}/{overall_total} = {overall_counts['mag_matches']/overall_total*100:.1f}%")
    print(f"All-match:       {overall_counts['all_matches']}/{overall_total} = {overall_counts['all_matches']/overall_total*100:.1f}%")
    print(f"\n(All-match = tag + type + direction + magnitude all correct)")

print(f"\nResults saved to: {output_dir}/")
print(f"  - {len(excel_rows_by_episode)} episode HTML visualizations")
print(f"  - operator_action_recommendations_by_episode.xlsx")
print(f"  - episode_summary.xlsx")

=== 4-Component Similarity + Sub-Minute Merging ===
Training lookup table: 361 entries from 86 episodes
Test episodes from 2025: 50
Processing 50 test episodes...
Output directory: /home/h604827/ControlActions/RESULTS/similarity_test_results/no_deviation_episodes_v3_merged



Test episodes: 100%|██████████| 50/50 [00:07<00:00,  6.41it/s]



OVERALL RESULTS (Sub-Minute Merged)
Tag match:       183/210 = 87.1%
Type match:      185/210 = 88.1%
Direction match: 115/210 = 54.8%
Magnitude match: 132/210 = 62.9%
All-match:       81/210 = 38.6%

(All-match = tag + type + direction + magnitude all correct)

Results saved to: /home/h604827/ControlActions/RESULTS/similarity_test_results/no_deviation_episodes_v3_merged/
  - 33 episode HTML visualizations
  - operator_action_recommendations_by_episode.xlsx
  - episode_summary.xlsx


## Test the Similarity Method on `history.parquet`

`history.parquet` is a short PV-only runtime window, not an episode/action dataset.

This test cell adapts it to the similarity pipeline by:
- loading the snapshot window and renaming `_PV` columns to `.PV`
- using the first timestamp as a proxy baseline for normalized ROC
- comparing each snapshot against the saved training lookup table
- returning the top historical action recommendation for each timestamp

Because this file does not contain operator actions or OP columns, this section produces recommendations only; it does not compute match accuracy.

In [4]:
# Step 26: Run the similarity recommender on DATA/history.parquet snapshots
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

history_path = Path('/home/h604827/ControlActions/DATA/history.parquet')
training_context_path = Path('/home/h604827/ControlActions/RESULTS/similarity_test_results/no_deviation_episodes_v3_merged/similarity_context_training_merged.csv')
operating_limits_path = Path('/home/h604827/ControlActions/DATA/operating_limits.csv')
history_output_path = Path('/home/h604827/ControlActions/RESULTS/similarity_test_results/no_deviation_episodes_v3_merged/history_similarity_recommendations.csv')
history_output_path.parent.mkdir(parents=True, exist_ok=True)

if not history_path.exists():
    raise FileNotFoundError(f'Missing history snapshot file: {history_path}')
if not training_context_path.exists():
    raise FileNotFoundError(
        'Training context CSV not found. Run the training-context export cell first or check the results folder.'
    )

if 'calculate_weighted_similarity' not in globals():
    def calculate_weighted_similarity(runtime_context, historical_df,
                                       norm_pos_cols, norm_roc_cols, dir_cols,
                                       w_norm_pos=0.25, w_norm_roc=0.30,
                                       w_alarm_prox=0.20, w_dir=0.25):
        if len(norm_pos_cols) == 0:
            raise ValueError('No shared normalized-position columns are available for similarity scoring.')

        runtime_norm_pos = np.array([runtime_context.get(col, 0.0) for col in norm_pos_cols])
        runtime_norm_pos = np.nan_to_num(runtime_norm_pos, nan=0.0)

        runtime_norm_roc = np.array([runtime_context.get(col, 0.0) for col in norm_roc_cols])
        runtime_norm_roc = np.nan_to_num(runtime_norm_roc, nan=0.0)

        runtime_dir = np.array([runtime_context.get(col, 0.0) for col in dir_cols])
        runtime_dir = np.nan_to_num(runtime_dir, nan=0.0)

        runtime_alarm_prox = runtime_context.get('alarm_proximity', 0.5)
        if pd.isna(runtime_alarm_prox):
            runtime_alarm_prox = 0.5

        hist_norm_pos = historical_df[norm_pos_cols].fillna(0.0).values
        hist_norm_roc = historical_df[norm_roc_cols].fillna(0.0).values
        hist_dir = historical_df[dir_cols].fillna(0.0).values
        hist_alarm_prox = historical_df['alarm_proximity'].fillna(0.5).values

        n_tags = len(norm_pos_cols)
        norm_pos_diffs = hist_norm_pos - runtime_norm_pos
        norm_pos_dists = np.sqrt(np.sum(norm_pos_diffs ** 2, axis=1))
        max_norm_pos_dist = np.sqrt(n_tags)
        norm_pos_sim = 1.0 - np.clip(norm_pos_dists / max_norm_pos_dist, 0, 1)

        norm_roc_diffs = hist_norm_roc - runtime_norm_roc
        norm_roc_dists = np.sqrt(np.sum(norm_roc_diffs ** 2, axis=1))
        max_norm_roc_dist = np.sqrt(n_tags * 4)
        norm_roc_sim = 1.0 - np.clip(norm_roc_dists / max_norm_roc_dist, 0, 1)

        alarm_prox_diffs = np.abs(hist_alarm_prox - runtime_alarm_prox)
        alarm_prox_sim = 1.0 - np.clip(alarm_prox_diffs, 0, 1)

        abs_runtime_roc = np.abs(runtime_norm_roc)
        abs_hist_roc = np.abs(hist_norm_roc)
        movement_weights = np.maximum(abs_runtime_roc, abs_hist_roc)
        dir_matches_matrix = (hist_dir == runtime_dir).astype(float)
        weight_sums = movement_weights.sum(axis=1)
        safe_weight_sums = np.where(weight_sums > 0, weight_sums, 1.0)
        weighted_dir_match = np.where(
            weight_sums > 0,
            (dir_matches_matrix * movement_weights).sum(axis=1) / safe_weight_sums,
            0.5
        )

        total_sim = (
            w_norm_pos * norm_pos_sim
            + w_norm_roc * norm_roc_sim
            + w_alarm_prox * alarm_prox_sim
            + w_dir * weighted_dir_match
        )

        num_raw_actions = historical_df['num_raw_actions'].fillna(1).values if 'num_raw_actions' in historical_df.columns else np.ones(len(historical_df))

        results = pd.DataFrame({
            'hist_index': historical_df.index,
            'total_similarity': total_sim,
            'norm_pos_similarity': norm_pos_sim,
            'norm_roc_similarity': norm_roc_sim,
            'alarm_prox_similarity': alarm_prox_sim,
            'weighted_dir_match': weighted_dir_match,
            'action_source': historical_df['action_source'].values,
            'action_type': historical_df['action_type'].values,
            'action_direction': historical_df['action_direction'].values,
            'action_magnitude': historical_df['action_magnitude'].values,
            'episode_id': historical_df['episode_id'].values,
            'num_raw_actions': num_raw_actions
        })

        return results.sort_values('total_similarity', ascending=False)

history_raw_df = pd.read_parquet(history_path)
history_df = history_raw_df.copy()
history_df['DateTime'] = pd.to_datetime(history_df['DateTime'], utc=True, errors='coerce')
if history_df['DateTime'].isna().any():
    raise ValueError('DateTime contains nulls after parsing. Clean the file before running this test.')

history_df['DateTime'] = history_df['DateTime'].dt.tz_convert(None)
history_df = history_df.rename(
    columns={col: col.replace('_PV', '.PV') for col in history_df.columns if col.endswith('_PV')}
 )
history_df = history_df.set_index('DateTime').sort_index()

training_context_df = pd.read_csv(training_context_path)
for dt_col in ['alarm_start', 'alarm_end', 'deviation_start', 'action_timestamp']:
    if dt_col in training_context_df.columns:
        training_context_df[dt_col] = pd.to_datetime(training_context_df[dt_col], errors='coerce')

runtime_limits_df = pd.read_csv(operating_limits_path)
history_op_limits = {}
for _, row in runtime_limits_df.iterrows():
    tag_base = row['TAG_NAME'].replace('.PV', '').replace('.OP', '')
    lower = row['LOWER_LIMIT']
    upper = row['UPPER_LIMIT']
    op_range = upper - lower
    if op_range > 0:
        history_op_limits[tag_base] = {'lower': lower, 'upper': upper, 'range': op_range}

alarm_threshold_history = globals().get('ALARM_THRESHOLD', 28.75)
target_upper_history = history_op_limits.get('03LIC_1071', {}).get('upper', 42.41)

episode_duration_minutes = np.nan
if {'episode_id', 'alarm_start', 'alarm_end'}.issubset(training_context_df.columns):
    episode_duration_minutes = (
        training_context_df[['episode_id', 'alarm_start', 'alarm_end']]
        .drop_duplicates()
        .assign(duration_minutes=lambda df: (df['alarm_end'] - df['alarm_start']).dt.total_seconds() / 60.0)['duration_minutes']
        .median()
    )
if pd.isna(episode_duration_minutes) or episode_duration_minutes <= 0:
    episode_duration_minutes = 60.0

history_pv_tags = [col for col in history_df.columns if col.endswith('.PV')]
history_tag_bases = {col.replace('.PV', '') for col in history_pv_tags}
training_tag_bases = {col.replace('_norm_pos', '') for col in training_context_df.columns if col.endswith('_norm_pos')}
shared_tag_bases = sorted(training_tag_bases & history_tag_bases)
missing_training_tags = sorted(training_tag_bases - history_tag_bases)

required_history_tags = {'03LIC_1071', '03LIC_1016', '03PIC_1013'}
missing_required_tags = sorted(required_history_tags - history_tag_bases)
if missing_required_tags:
    raise ValueError(f'history.parquet is missing required target PV tags: {missing_required_tags}')

shared_norm_pos_cols = [f'{tag}_norm_pos' for tag in shared_tag_bases if f'{tag}_norm_pos' in training_context_df.columns]
shared_norm_roc_cols = [f'{tag}_norm_roc' for tag in shared_tag_bases if f'{tag}_norm_roc' in training_context_df.columns]
shared_dir_cols = [f'{tag}_roc_direction' for tag in shared_tag_bases if f'{tag}_roc_direction' in training_context_df.columns]

if not shared_norm_pos_cols:
    raise ValueError('No shared PV tags were found between history.parquet and the saved training context.')

requested_history_start_time = pd.Timestamp('06:24:08').time()
history_eval_start = history_df.index.min().normalize() + pd.Timedelta(hours=6, minutes=24, seconds=8)
history_eval_df = history_df.loc[history_df.index >= history_eval_start].copy()
if history_eval_df.empty:
    raise ValueError(f'No history snapshots found at or after {history_eval_start}.')

print('=== history.parquet profile ===')
print(f'Full shape: {history_raw_df.shape}')
print(f'Full time range (UTC-naive): {history_df.index.min()} -> {history_df.index.max()}')
print(f'Evaluation start time: {history_eval_start} (requested cutoff {requested_history_start_time})')
print(f'Evaluation rows: {len(history_eval_df)} / {len(history_df)}')
print(f'Evaluation time range: {history_eval_df.index.min()} -> {history_eval_df.index.max()}')
print(f'PV tags in history file: {len(history_pv_tags)}')
print(f'Shared PV tags with training context: {len(shared_tag_bases)}')
print(f'Approximate typical training episode duration: {episode_duration_minutes:.1f} minutes')
if missing_training_tags:
    print(f'Training tags not present in history snapshots ({len(missing_training_tags)}): {missing_training_tags}')

history_start = history_eval_df.index.min()
history_baseline = history_eval_df.loc[history_start]

def build_history_runtime_context(current_time):
    current_row = history_eval_df.loc[current_time]
    context = {}

    for tag_base in shared_tag_bases:
        pv_tag = f'{tag_base}.PV'
        norm_pos_col = f'{tag_base}_norm_pos'
        norm_roc_col = f'{tag_base}_norm_roc'
        dir_col = f'{tag_base}_roc_direction'

        limits = history_op_limits.get(tag_base)
        current_val = current_row.get(pv_tag, np.nan)
        start_val = history_baseline.get(pv_tag, np.nan)

        if limits and pd.notna(current_val):
            norm_pos = (current_val - limits['lower']) / limits['range']
        else:
            norm_pos = 0.0

        if limits and pd.notna(start_val) and pd.notna(current_val):
            norm_roc = (current_val - start_val) / limits['range']
        else:
            norm_roc = 0.0

        if norm_pos_col in shared_norm_pos_cols:
            context[norm_pos_col] = norm_pos
        if norm_roc_col in shared_norm_roc_cols:
            context[norm_roc_col] = norm_roc
        if dir_col in shared_dir_cols:
            context[dir_col] = 1 if norm_roc >= 0 else 0

    pv_1071 = current_row.get('03LIC_1071.PV', np.nan)
    if pd.notna(pv_1071):
        context['alarm_proximity'] = (pv_1071 - alarm_threshold_history) / (target_upper_history - alarm_threshold_history)
    else:
        context['alarm_proximity'] = 0.5

    minutes_from_start = (current_time - history_start).total_seconds() / 60.0
    context['time_progress'] = minutes_from_start / episode_duration_minutes

    return context

history_results = []
for current_time in history_eval_df.index:
    runtime_context = build_history_runtime_context(current_time)
    sim_results = calculate_weighted_similarity(
        runtime_context,
        training_context_df,
        shared_norm_pos_cols,
        shared_norm_roc_cols,
        shared_dir_cols
    )

    top_match = sim_results.iloc[0]
    history_results.append({
        'history_time': current_time,
        '03LIC_1071.PV': history_eval_df.at[current_time, '03LIC_1071.PV'],
        '03LIC_1016.PV': history_eval_df.at[current_time, '03LIC_1016.PV'],
        '03PIC_1013.PV': history_eval_df.at[current_time, '03PIC_1013.PV'],
        'alarm_proximity': runtime_context['alarm_proximity'],
        'time_progress': runtime_context['time_progress'],
        'top_similarity': top_match['total_similarity'],
        'norm_pos_similarity': top_match['norm_pos_similarity'],
        'norm_roc_similarity': top_match['norm_roc_similarity'],
        'alarm_prox_similarity': top_match['alarm_prox_similarity'],
        'weighted_dir_match': top_match['weighted_dir_match'],
        'recommended_action_source': top_match['action_source'],
        'recommended_action_type': top_match['action_type'],
        'recommended_action_direction': int(top_match['action_direction']),
        'recommended_action_magnitude': float(top_match['action_magnitude']),
        'matched_training_episode_id': top_match['episode_id'],
        'matched_num_raw_actions': int(top_match['num_raw_actions'])
    })

history_similarity_df = pd.DataFrame(history_results)
history_similarity_df.to_csv(history_output_path, index=False)

history_recommendation_counts_df = (
    history_similarity_df
    .groupby(['recommended_action_source', 'recommended_action_type', 'recommended_action_direction'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
 )

print(f'History recommendations saved to: {history_output_path}')
print(f'Total snapshots scored: {len(history_similarity_df)}')

display(history_similarity_df)
display(history_recommendation_counts_df.head(10))

history_similarity_fig = go.Figure()
history_similarity_fig.add_trace(
    go.Scatter(
        x=history_similarity_df['history_time'],
        y=history_similarity_df['03LIC_1071.PV'],
        mode='lines+markers',
        name='03LIC_1071.PV'
    )
)
history_similarity_fig.add_trace(
    go.Scatter(
        x=history_similarity_df['history_time'],
        y=history_similarity_df['top_similarity'],
        mode='lines+markers',
        name='Top similarity',
        yaxis='y2'
    )
)
history_similarity_fig.update_layout(
    title='history.parquet: Target PV vs. Top Similarity Score',
    xaxis_title='Snapshot time',
    yaxis=dict(title='03LIC_1071.PV'),
    yaxis2=dict(title='Top similarity', overlaying='y', side='right', range=[0, 1]),
    height=450
 )
history_similarity_fig.show()

=== history.parquet profile ===
Full shape: (120, 28)
Full time range (UTC-naive): 2026-04-27 05:14:08 -> 2026-04-27 07:13:08
Evaluation start time: 2026-04-27 06:24:08 (requested cutoff 06:24:08)
Evaluation rows: 50 / 120
Evaluation time range: 2026-04-27 06:24:08 -> 2026-04-27 07:13:08
PV tags in history file: 27
Shared PV tags with training context: 27
Approximate typical training episode duration: 4.0 minutes
Training tags not present in history snapshots (1): ['02FI_1000']
History recommendations saved to: /home/h604827/ControlActions/RESULTS/similarity_test_results/no_deviation_episodes_v3_merged/history_similarity_recommendations.csv
Total snapshots scored: 50


,history_time,03LIC_1071.PV,03LIC_1016.PV,03PIC_1013.PV,alarm_proximity,time_progress,top_similarity,norm_pos_similarity,norm_roc_similarity,alarm_prox_similarity,weighted_dir_match,recommended_action_source,recommended_action_type,recommended_action_direction,recommended_action_magnitude,matched_training_episode_id,matched_num_raw_actions
0,2026-04-27 06:24:08,38.792492,36.975231,1.329563,0.734914,0.00,0.670008,0.0,0.860515,0.923315,0.908762,03PIC_1013,OP,0,-4.0,383,2
1,2026-04-27 06:25:08,37.291489,34.828220,1.304130,0.625070,0.25,0.447760,0.0,0.000000,0.988859,0.999952,03PIC_1013,OP,1,2.0,56,1
2,2026-04-27 06:26:08,35.464828,33.240585,1.291963,0.491394,0.50,0.448215,0.0,0.000000,0.991127,0.999959,03PIC_1013,OP,1,2.0,119,1
3,2026-04-27 06:27:08,33.423508,32.211170,1.290700,0.342009,0.75,0.449480,0.0,0.000000,0.997479,0.999939,03PIC_1013,OP,0,-2.0,173,2
4,2026-04-27 06:28:08,31.253973,31.703028,1.296784,0.183242,1.00,0.447500,0.0,0.000000,0.987539,0.999969,03PIC_1013,OP,0,-2.0,132,1
5,2026-04-27 06:29:08,29.006407,31.765848,1.304801,0.018764,1.25,0.448364,0.0,0.000000,0.991874,0.999956,03PIC_1013,OP,0,-1.0,173,1
6,2026-04-27 06:30:08,26.822300,32.366173,1.310075,-0.141070,1.50,0.448481,0.0,0.000000,0.992440,0.999971,03PIC_1013,OP,0,-2.0,447,1
7,2026-04-27 06:31:08,25.543468,32.898754,1.311849,-0.234655,1.75,0.445175,0.0,0.000000,0.975887,0.999992,03LIC_1071,SP,1,1.0,271,1
8,2026-04-27 06:32:08,24.292133,33.498478,1.312655,-0.326229,2.00,0.426861,0.0,0.000000,0.884314,0.999993,03LIC_1071,SP,1,1.0,271,1
9,2026-04-27 06:33:08,21.747639,34.787323,1.312169,-0.512436,2.25,0.389619,0.0,0.000000,0.698106,0.999991,03LIC_1071,SP,1,1.0,271,1


,recommended_action_source,recommended_action_type,recommended_action_direction,count
0,03LIC_1071,OP,0,17
1,03PIC_1013,OP,0,15
2,03LIC_1071,OP,1,8
3,03LIC_1071,SP,1,7
4,03PIC_1013,OP,1,3
